In [1]:
import polars as pl

In [2]:
books_df = pl.scan_ndjson("../processed-data/cleaned_books_fantasy_paranormal.json")
interactions_df = pl.scan_ndjson("../processed-data/cleaned_interactions_fantasy_paranormal.json")
authors_df = pl.scan_ndjson("../raw-data/goodreads_book_authors.json")
series_df = pl.scan_ndjson("../raw-data/goodreads_book_series.json")

# Ensure join keys have consistent dtypes (some GoodReads dumps mix int/str ids)
books_df = books_df.with_columns(
    pl.col("book_id").cast(pl.Int64, strict=False),
    pl.col("work_id").cast(pl.Int64, strict=False),
    pl.col("ratings_count").cast(pl.Int64, strict=False),
)
interactions_df = interactions_df.with_columns(
    pl.col("book_id").cast(pl.Int64, strict=False)
)

In [3]:
# filter out descriptions with fewer than 50 words
books_df = books_df.filter(pl.col('description').str.count_matches(r"\w+") >= 50)

In [4]:
books_df.select('language_code').unique().show()

language_code
str
"""fin"""
"""dan"""
"""grc"""
"""heb"""
"""lat"""


In [5]:
# Keep only english variations
# """en"""
# """en-CA"""
# """en-GB"""
# """en-US"""
# """eng"""

books_df = books_df.filter(
    pl.col('language_code').is_in(['en', 'en-CA', 'en-GB', 'en-US', 'eng'])
)


In [6]:
# Deduplicate books by work_id (keep a single representative edition per work).
# Heuristic: prefer higher ratings_count, then longer description, then lower book_id for determinism.
books_df = (
    books_df
    .with_columns(
        pl.col("description").fill_null("").str.len_chars().alias("_desc_len")
    )
    .sort(["work_id", "ratings_count", "_desc_len", "book_id"], descending=[False, True, True, False])
    .unique(subset=["work_id"], keep="first")
    .drop(["_desc_len"])
)

In [7]:
books_df = books_df.explode("authors").with_columns(
    pl.col("authors").struct.field("author_id").alias("author_id")
)
books_df = books_df.join(
    authors_df.select(["author_id", "name"]),
    on="author_id",
    how="left"
).group_by("book_id").agg(
    pl.all().exclude(["authors", "author_id", "name"]).first(),
    pl.col("name").drop_nulls().alias("author_names")
)


In [8]:
books_df = books_df.explode("series").with_columns(
    pl.col("series").alias("series_id")
)
books_df = books_df.join(
    series_df.select(["series_id", pl.col("title").alias("series_title")]),
    on="series_id",
    how="left"
).group_by("book_id").agg(
    pl.all().exclude(["series", "series_id", "series_title"]).first(),
    pl.col("series_title").drop_nulls().alias("series")
)


In [9]:
books_df = books_df.with_columns(
    pl.col("popular_shelves").list.eval(
        pl.element().struct.field("name")
    ).alias("popular_shelves")
).with_columns(
    pl.col("popular_shelves").list.eval(
        pl.element().filter(
            ~pl.element().str.contains(r"(?i)read|own|buy|fav|library|audio|kindle|ebook")
        )
    )
)


In [10]:
books_df.select('language_code').unique().show()

language_code
str
"""en-GB"""
"""en-CA"""
"""eng"""
"""en"""
"""en-US"""


In [11]:
books_df.head().show()

book_id,isbn,text_reviews_count,country_code,language_code,popular_shelves,asin,is_ebook,average_rating,kindle_asin,similar_books,description,format,link,publisher,num_pages,publication_day,isbn13,publication_month,edition_information,publication_year,url,image_url,ratings_count,work_id,title,title_without_series,author_names,series
i64,str,i64,str,str,list[str],str,str,f64,str,list[str],str,str,str,str,i64,i64,str,i64,str,i64,str,str,i64,i64,str,str,list[str],list[str]
13645297,"""0753540991""",6,"""US""","""eng""","[""fantasy"", ""humor"", … ""parodies-humor""]","""""","""false""",2.28,"""""","[""12998073"", ""13129857"", … ""13167162""]","""'There was a bloodied knife on…","""Paperback""","""https://www.goodreads.com/book…","""Virgin Books (Ebury Publishing…",228,null,"""9780753540992""",null,"""""",2012,"""https://www.goodreads.com/book…","""https://images.gr-assets.com/b…",17,17827694,"""A Game of Groans: A Parody of …","""A Game of Groans: A Parody of …","[""George R.R. Washington""]",[]
23279680,"""""",186,"""US""","""en-US""","[""paranormal"", ""ghosts"", … ""cozy-mysteries""]","""B00NO4S94I""","""true""",4.06,"""B00NO4S94I""","[""17731924"", ""13088838"", … ""18275071""]","""Ellie Jordan's job is to catch…","""""","""https://www.goodreads.com/book…","""""",null,null,"""""",null,"""""",null,"""https://www.goodreads.com/book…","""https://images.gr-assets.com/b…",1787,42698680,"""Ellie Jordan, Ghost Trapper (E…","""Ellie Jordan, Ghost Trapper (E…","[""J.L. Bryan""]","[""Ellie Jordan, Ghost Trapper""]"
17232049,"""""",47,"""US""","""eng""","[""series"", ""fantasy"", … ""my-books""]","""""","""false""",4.52,"""""",[],"""High up in the tallest tower o…","""Paperback""","""https://www.goodreads.com/book…","""Evolved Publishing""",null,19,"""""",3,"""""",2013,"""https://www.goodreads.com/book…","""https://images.gr-assets.com/b…",61,23748208,"""The Persnickety Princess (Tale…","""The Persnickety Princess (Tale…","[""Falcon Storm""]","[""Tales from Upon A. Time""]"
23432918,"""""",17,"""US""","""eng""","[""young-adult"", ""paranormal"", … ""paranormal-fantasy""]","""""","""false""",4.25,"""""",[],"""Questions, questions, and more…","""Paperback""","""https://www.goodreads.com/book…","""Booktrope""",264,21,"""9781620155592""",10,"""""",2014,"""https://www.goodreads.com/book…","""https://images.gr-assets.com/b…",43,42992982,"""Perception (Eve #3)""","""Perception (Eve #3)""","[""A.L. Waddington""]","[""Eve""]"
26862475,"""""",1,"""US""","""eng""","[""romance"", ""paranormal"", … ""pending-placement""]","""B0163KAD20""","""true""",2.93,"""B01IW4P8BQ""",[],"""**A Fun, Standalone HIGHLANDER…","""""","""https://www.goodreads.com/book…","""""",null,null,"""""",null,"""""",null,"""https://www.goodreads.com/book…","""https://s.gr-assets.com/assets…",21,46902895,"""Into the Highlander's Arms""","""Into the Highlander's Arms""","[""Samantha Leal""]",[]


In [12]:
books_df = books_df.drop([
    'isbn', 'text_reviews_count', 'country_code', 'language_code',
    'is_ebook', 'kindle_asin', 'format', 'num_pages',
    'publication_day', 'isbn13', 'publication_month', 'edition_information', 'publication_year', 'image_url',
    'title_without_series', 'ratings_count', 'publisher', 'similar_books', 'asin'
])
books_df.head().show()


book_id,popular_shelves,average_rating,description,link,url,work_id,title,author_names,series
i64,list[str],f64,str,str,str,i64,str,list[str],list[str]
12415904,"[""steampunk"", ""m-m"", … ""historical-romance""]",3.13,"""If you build it, love will com…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",17397731,"""Far Too Human""","[""Anitra Lynn McLeod""]",[]
10536995,"[""m-m"", ""paranormal"", … ""z-wc-20-40""]",3.97,"""Shane never expected to do wha…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",15443348,"""Shane's Fury (Lost Shifters #1…","[""Stephani Hecht""]","[""Lost Shifters""]"
17852846,"[""m-m"", ""vampires"", … ""3-stars""]",3.98,"""[Menage Amour ManLove: Erotic …","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",24989297,"""Virgil (Marius Brothers #8)""","[""Joyee Flynn""]","[""Marius Brothers"", ""Marius World""]"
16079583,"[""science-fiction"", ""ptntl-srs-seqs"", … ""scifi-freebies""]",4.83,"""Second book in Starbirth serie…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",21877514,"""The Shifter Dimension (Starbir…","[""J.M. Johnson""]","[""Starbirth""]"
16066215,"[""aliens"", ""dragons"", … ""august-2017""]",4.19,"""Ariel Hamm has always had a te…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",21857208,"""Ambushing Ariel (Dragon Lords …","[""S.E. Smith""]","[""Dragon Lords of Valdier""]"


In [13]:
books_df = books_df.with_columns(
    pl.concat_str(
        [
            pl.col("title").fill_null(""),
            pl.col("title").fill_null(""),
            pl.col("title").fill_null(""),
            pl.col("series").list.join(", ").fill_null(""),
            pl.col("popular_shelves").list.join(", ").fill_null(""),
            pl.col("description").fill_null("")
        ],
        separator=" "
    ).alias("combined_text")
)


In [14]:
# Basic text preprocessing on combined_text
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

STOPWORDS = set(stopwords.words('english'))
stopwords_regex = r"(?i)\b(" + "|".join(STOPWORDS) + r")\b"

lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    if not text:
        return ""
    return " ".join([lemmatizer.lemmatize(w) for w in text.split()])

books_df = books_df.with_columns(
    pl.col("combined_text")
    .str.to_lowercase()
    .str.replace_all(r"[^\w\s]", " ") # remove punctuation
    .str.replace_all(stopwords_regex, "") # remove stopwords
    .str.replace_all(r"\s+", " ") # remove multiple spaces
    .str.strip_chars() # trim leading/trailing spaces
    .map_elements(lemmatize_text, return_dtype=pl.String)
)

books_df.head().show()


book_id,popular_shelves,average_rating,description,link,url,work_id,title,author_names,series,combined_text
i64,list[str],f64,str,str,str,i64,str,list[str],list[str],str
15808242,"[""fantasy"", ""fiction"", … ""ya""]",3.47,"""Cosmically fast-paced and wild…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",21505929,"""City of Dark Magic (City of Da…","[""Magnus Flyte""]","[""City of Dark Magic""]","""city dark magic city dark magi…"
62544,"[""horror"", ""thriller"", … ""_reviewed""]",4.15,"""Can you imagine a new chemical…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",60746,"""All the Rage (Repairman Jack, …","[""F. Paul Wilson""]","[""The Secret History of the World"", ""Repairman Jack""]","""rage repairman jack 4 rage rep…"
751369,"[""fantasy"", ""christian-fiction"", … ""classics""]",4.32,"""The gallows loom large over Ti…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",737502,"""The Golden Wood (The King of t…","[""William D. Burt""]","[""The King of the Trees""]","""golden wood king tree 3 golden…"
22401306,"[""paranormal"", ""freebie"", … ""romantic-suspence""]",4.11,"""Detective Jordan Delany has a …","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",41825864,"""Dream Huntress (Dream Seeker, …","[""Michelle Sharp""]","[""Dream Seeker""]","""dream huntress dream seeker 1 …"
26106271,"[""paranormal"", ""romance"", … ""2-bon-marché""]",4.11,"""All Jet Taylor wanted was to s…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",46050866,"""Outlaw Bear (Bluff Bears #2)""","[""Amelia Jade"", ""Terra Wolf""]","[""Bluff Bears""]","""outlaw bear bluff bear 2 outla…"


In [15]:
import os
os.makedirs("../processed-data", exist_ok=True)
books_df.collect().write_ndjson("../processed-data/processed_books_texts.json")
print("Dataset successfully saved!")

Dataset successfully saved!
